# Roboto Origin forward-only curriculum

Pinned task source: 36521e4a473595ccc08ad73fb99113f408ef4960. GPU runtime and authorized group Drive required. Trains only forward walking, not box transfer. Up to 4 x 500 updates with 512 environments, with a frozen 5-trial evaluation after each chunk. It stops early if the internal one-meter progress gate passes; video still needs review. Resumes the latest forward-family checkpoint, falling back to the preserved balance checkpoint. Training defaults off.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
"""Colab recovery run; downloads its backup immediately after training finishes.

EULA acceptance was explicitly authorized by Zhao-Yu in this conversation.
This script uses only the group's public source and its own generated files.
"""

import os
import re
import subprocess
import sys
import time
from pathlib import Path

PYTHON = '/content/rpo-venv/bin/python'
LAB = Path('/content/IsaacLab')
REPO = Path('/content/LearningRL')
REVISION = '36521e4a473595ccc08ad73fb99113f408ef4960'


def checked(args, cwd=None, timeout=600):
    p = subprocess.run(args, cwd=cwd, text=True, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, timeout=timeout)
    print(p.stdout[-3500:], flush=True)
    p.check_returncode()


checked([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if not Path(PYTHON).is_file():
    checked(['uv', 'venv', '--python', '3.12', '/content/rpo-venv'])
if not LAB.exists():
    checked(['git', 'clone', '--depth', '1', '--branch', 'v3.0.0-beta2.patch1',
             'https://github.com/isaac-sim/IsaacLab.git', str(LAB)])
if not REPO.exists():
    checked(['git', 'clone', '--branch', 'student/zhaoyu-lu-HMD', '--single-branch',
             'https://github.com/yuki-taiwanese/LearningRL.git', str(REPO)])
checked(['git', '-C', str(REPO), 'checkout', REVISION])
override = Path('/content/rpo-overrides.txt')
override.write_text('coverage==7.4.4\npackaging==26.0\n')
args = ['uv', 'pip', 'install', '--python', PYTHON, '--extra-index-url',
        'https://pypi.nvidia.com', '--index-strategy', 'unsafe-best-match',
        '--prerelease', 'allow', '--override', str(override),
        'isaacsim[all,extscache]==6.0.1.0', 'wandb', 'ruff']
for package in ['isaaclab', 'isaaclab_assets', 'isaaclab_contrib',
                'isaaclab_newton[all]', 'isaaclab_physx[newton]', 'isaaclab_rl[rsl-rl]',
                'isaaclab_tasks', 'isaaclab_ov', 'isaaclab_ovphysx',
                'isaaclab_ppisp', 'isaaclab_visualizers[kit]']:
    name, _, extras = package.partition('[')
    args += ['-e', str(LAB / 'source' / name) + ('[' + extras if extras else '')]
args += ['-e', str(REPO / 'robot_description'), '-e', str(REPO / 'LearningRLlab')]
print('Installing the matched simulation environment...', flush=True)
checked(args, timeout=900)

asset = REPO / 'robot_description/vendor/rpo_description'
asset.parent.mkdir(parents=True, exist_ok=True)
if not asset.exists():
    checked(['git', 'clone', 'https://github.com/Roboparty/rpo_description.git', str(asset)])
checked(['git', '-C', str(asset), 'checkout', '37aac9ca665e92731444a1618320078e7ba21569'])

env = os.environ.copy()
env.update(OMNI_KIT_ACCEPT_EULA='YES', ISAACLAB_PATH=str(LAB), WANDB_MODE='disabled',
           MPLBACKEND='Agg', PYTHONUNBUFFERED='1')
model_path = Path('/content/rpo-converted-path.txt')
if not model_path.exists():
    # Import with Kit once; training itself never uses a Vulkan renderer.
    conversion = r'''
import os
from pathlib import Path
from isaacsim import SimulationApp
app = SimulationApp({'headless': True})
from isaaclab.sim.converters import UrdfConverter
from robot_description.robots.roboparty_rpo import RPO_CFG
from robot_description.tools.usd_inertia_fix import fix_principal_axes
loader = UrdfConverter(RPO_CFG.spawn)
fix_principal_axes(loader.usd_path)
Path('/content/rpo-converted-path.txt').write_text(loader.usd_path)
print('CONVERTED_AND_INERTIA_FIXED:', loader.usd_path, flush=True)
# The converter has synchronously saved all layers. Avoid Colab's Kit shutdown crash.
os._exit(0)
'''
    with Path('/content/rpo-conversion.log').open('w') as log:
        p = subprocess.run([PYTHON, '-c', conversion], env=env, stdout=log,
                           stderr=subprocess.STDOUT, timeout=240)
    print(Path('/content/rpo-conversion.log').read_text()[-3500:], flush=True)
    assert model_path.exists(), 'Robot conversion did not complete; inspect rpo-conversion.log'

env['LEARNINGRL_RPO_USD'] = model_path.read_text().strip()
assert Path(env['LEARNINGRL_RPO_USD']).is_file()


In [ ]:
from pathlib import Path
Path('/content/evaluate_walk.py').write_text('"""Five independent nominal 10-second walking trials, using a frozen policy."""\nimport argparse\nimport importlib.metadata\nimport json\nimport sys\nfrom pathlib import Path\n\nimport warp as wp\nwp.config.enable_backward = False\nimport gymnasium as gym\nimport torch\nimport numpy as np\nfrom rsl_rl.runners import OnPolicyRunner\nimport LearningRLlab.tasks\nfrom isaaclab_rl.rsl_rl import RslRlVecEnvWrapper, handle_deprecated_rsl_rl_cfg\nfrom isaaclab_tasks.utils import add_launcher_args, launch_simulation, setup_preset_cli\nfrom isaaclab_tasks.utils.hydra import hydra_task_config\n\nparser = argparse.ArgumentParser()\nparser.add_argument(\'--task\', default=\'LearningRL-RPO-FlatWalk-v0-PLAY\')\nparser.add_argument(\'--checkpoint\', required=True)\nparser.add_argument(\'--output\', required=True)\nparser.add_argument(\'--forward-only\', action=\'store_true\')\nparser.add_argument(\'--trace\', action=\'store_true\')\nadd_launcher_args(parser)\nargs, remaining = setup_preset_cli(parser)\nsys.argv = [sys.argv[0]] + remaining\n\n\n@hydra_task_config(args.task, \'rsl_rl_cfg_entry_point\')\ndef main(cfg, agent_cfg):\n    agent_cfg = handle_deprecated_rsl_rl_cfg(agent_cfg, importlib.metadata.version(\'rsl-rl-lib\'))\n    cfg.scene.num_envs = 5\n    cfg.seed = 20261116\n    cfg.observations.policy.enable_corruption = False\n    cfg.commands.base_velocity.resampling_time_range = (1000., 1000.)\n    cfg.commands.base_velocity.heading_command = False\n    cfg.commands.base_velocity.rel_standing_envs = 0.\n    with launch_simulation(cfg, {\'visualizer\': None, \'visualizer_explicit\': True}):\n        raw = gym.make(args.task, cfg=cfg)\n        env = RslRlVecEnvWrapper(raw, clip_actions=agent_cfg.clip_actions)\n        runner = OnPolicyRunner(env, agent_cfg.to_dict(), log_dir=None, device=agent_cfg.device)\n        runner.load(args.checkpoint)\n        policy = runner.get_inference_policy(device=env.unwrapped.device)\n        base = env.unwrapped\n        command = base.command_manager.get_term(\'base_velocity\')\n        robot = base.scene[\'robot\']\n        dt = base.step_dt\n        audit = []\n        reference_action = None\n        with torch.inference_mode():\n            for target_x in (0., .2, -.2):\n                command.vel_command_b[:] = torch.tensor((target_x, 0., 0.), device=base.device)\n                base.observation_manager.compute(update_history=True)\n                probe = env.get_observations()\n                observed = probe[\'policy\'][:, 6:9]\n                assert torch.allclose(observed, command.vel_command_b), \'Command is not present at actor inputs 6:9\'\n                act = policy(probe).clone()\n                if reference_action is None:\n                    reference_action = act\n                audit.append({\'requested_vx\':target_x,\'observed_command\':observed[0].cpu().tolist(),\'max_action_difference_from_stand\':float((act-reference_action).abs().max())})\n        assert max(row[\'max_action_difference_from_stand\'] for row in audit)>1e-6, \'Policy ignores velocity commands completely\'\n        alive = torch.ones(5, dtype=torch.bool, device=base.device)\n        survival = torch.full((5,), 10., device=base.device)\n        error = torch.zeros(5, device=base.device)\n        count = torch.zeros(5, device=base.device)\n        phase_velocity = torch.zeros((4, 5, 3), device=base.device)\n        phase_count = torch.zeros((4, 5), device=base.device)\n        initial_xy = robot.data.root_pos_w[:, :2].clone()\n        path_length = torch.zeros(5, device=base.device)\n        trace = []\n        with torch.inference_mode():\n            for step in range(round(10. / dt)):\n                t = step * dt\n                phase = 0 if t < 1 else (1 if t < 4 else (2 if t < 7 else 3))\n                target = (0., 0., 0.) if t < 1 else ((.2, 0., 0.) if t < 4 else ((0., .15, 0.) if t < 7 else (-.15, 0., 0.)))\n                if args.forward_only:\n                    phase = 0 if t < 1 else 1\n                    target = (0., 0., 0.) if t < 1 else (.2, 0., 0.)\n                command.vel_command_b[:] = torch.tensor(target, device=base.device)\n                # Refresh observations after setting the command, before inference.\n                base.observation_manager.compute(update_history=True)\n                obs = env.get_observations()\n                action = policy(obs)\n                error += torch.linalg.vector_norm(robot.data.root_lin_vel_b[:, :2] - command.vel_command_b[:, :2], dim=1) * alive\n                count += alive\n                phase_velocity[phase, :, :2] += robot.data.root_lin_vel_b[:, :2] * alive[:, None]\n                phase_velocity[phase, :, 2] += robot.data.root_ang_vel_b[:, 2] * alive\n                phase_count[phase] += alive\n                before_xy = robot.data.root_pos_w[:, :2].clone()\n                obs, _, dones, _ = env.step(action)\n                path_length += torch.linalg.vector_norm(robot.data.root_pos_w[:, :2] - before_xy, dim=1) * (alive & ~dones.bool())\n                newly_done = alive & dones.bool()\n                survival[newly_done] = (step + 1) * dt\n                alive &= ~dones.bool()\n                policy.reset(dones)\n                if args.trace:\n                    trace.append(torch.cat((robot.data.root_pos_w[0],robot.data.root_quat_w[0],robot.data.joint_pos[0])).cpu().numpy().copy())\n        result = {\n            \'checkpoint\': args.checkpoint, \'seed\': cfg.seed,\n            \'protocol\': \'Five nominal trials: stand 1s, forward 3s, lateral 3s, backward 3s. Stop scoring each trial at its first termination.\',\n            \'survived_10_seconds\': int(alive.sum()), \'trials\': 5,\n            \'survival_seconds\': survival.cpu().tolist(),\n            \'mean_xy_velocity_error_m_s\': (error / count.clamp_min(1)).cpu().tolist(),\n            \'stationary_reference_mean_xy_error_m_s\': 0.15,\n            \'phase_names\': [\'stand\', \'forward\', \'lateral\', \'backward\'],\n            \'phase_command_vx_vy_wz\': [[0., 0., 0.], [.2, 0., 0.], [0., .15, 0.], [-.15, 0., 0.]],\n            \'phase_mean_actual_vx_vy_wz_per_trial\': (phase_velocity / phase_count.clamp_min(1)[:, :, None]).cpu().tolist(),\n            \'xy_path_length_m\': path_length.cpu().tolist(),\n            \'net_xy_displacement_m_survivors\': torch.linalg.vector_norm(robot.data.root_pos_w[:, :2] - initial_xy, dim=1)[alive].cpu().tolist(),\n            \'box_transfer_success_established\': False,\n            \'command_input_audit\': audit,\n            \'forward_only\': args.forward_only,\n        }\n        if args.forward_only:\n            result[\'protocol\'] = \'Five nominal trials: stand 1s, forward 0.2 m/s for 9s. Score until first termination.\'\n            result[\'stationary_reference_mean_xy_error_m_s\'] = .18\n        if args.trace:\n            np.savez(Path(args.output).with_suffix(\'.npz\'), states=np.asarray(trace), dt=dt, joint_names=np.array(robot.joint_names), env_origin=base.scene.env_origins[0].cpu().numpy(), survived=bool(alive[0]))\n        Path(args.output).write_text(json.dumps(result, indent=2))\n        print(\'WALK_EVALUATION:\', json.dumps(result, indent=2), flush=True)\n        env.close()\n\n\nif __name__ == \'__main__\':\n    main()\n')
Path('/content/render_trace.py').write_text('"""CPU kinematic visualization of recorded Newton states, not another physics simulation."""\nimport argparse\nimport subprocess\nimport xml.etree.ElementTree as ET\nfrom pathlib import Path\nimport mujoco\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\nparser=argparse.ArgumentParser()\nparser.add_argument(\'trace\')\nargs=parser.parse_args()\npath=Path(args.trace)\ntrace=np.load(path)\nsource=Path(\'/content/LearningRL/robot_description/vendor/rpo_description/mjcf/rpo.xml\')\nroot=ET.parse(source).getroot()\nroot.find(\'compiler\').set(\'meshdir\',str(source.parent.parent/\'meshes\'))\nprint(\'CPU_TRACE: loading model\',flush=True)\nmodel=mujoco.MjModel.from_xml_string(ET.tostring(root,encoding=\'unicode\'))\ndata=mujoco.MjData(model)\nstates=trace[\'states\']; origin=trace[\'env_origin\']; dt=float(trace[\'dt\'])\nbase=model.joint(\'floating_base_joint\').qposadr[0]\nindices=[model.joint(str(n)).qposadr[0] for n in trace[\'joint_names\']]\nnames=[mujoco.mj_id2name(model,mujoco.mjtObj.mjOBJ_BODY,i) or \'\' for i in range(model.nbody)]\ncenter=np.array([states[0,0]-origin[0]+.55,states[0,1]-origin[1],.65])\nright=np.array([.8,-.6,0]); up=np.array([-.15,-.2,.9682458])\ndef project(point):\n    p=np.asarray(point)-center\n    return (int(320+np.dot(p,right)*200),int(180-np.dot(p,up)*200))\nstride=max(1,round(.1/dt)); fps=1/(stride*dt)\nvideo=path.with_suffix(\'.mp4\')\nlog=video.with_suffix(\'.render.log\').open(\'w\')\nencoder=subprocess.Popen([\'/usr/bin/ffmpeg\',\'-y\',\'-loglevel\',\'error\',\'-f\',\'rawvideo\',\'-pix_fmt\',\'rgb24\',\'-s\',\'640x360\',\'-r\',str(fps),\'-i\',\'-\',\'-an\',\'-c:v\',\'libx264\',\'-pix_fmt\',\'yuv420p\',str(video)],stdin=subprocess.PIPE,stdout=subprocess.DEVNULL,stderr=log)\nsamples=[]\nsample_indices=[min(len(states)-1,round(t/dt)//stride*stride) for t in (0,3,6,9)]\nprint(\'CPU_TRACE: rendering frames\',flush=True)\nfor i in range(0,len(states),stride):\n    s=states[i]\n    data.qpos[base:base+3]=s[:3]-origin\n    data.qpos[base+3:base+7]=s[3:7]\n    data.qpos[indices]=s[7:]\n    mujoco.mj_kinematics(model,data)\n    frame=Image.new(\'RGB\',(640,360),\'#eef2f5\')\n    draw=ImageDraw.Draw(frame)\n    for x in np.arange(-.5,3.,.5):\n        draw.line([project((x,-1,0)),project((x,1,0))],fill=\'#c3ced8\',width=1)\n    for y in np.arange(-1,1.1,.5):\n        draw.line([project((-.5,y,0)),project((2.5,y,0))],fill=\'#c3ced8\',width=1)\n    for b in range(1,model.nbody):\n        parent=model.body_parentid[b]\n        if parent==0:continue\n        color=\'#2166ac\' if \'left\' in names[b] else (\'#1b9e77\' if \'right\' in names[b] else \'#555555\')\n        draw.line([project(data.xpos[parent]),project(data.xpos[b])],fill=color,width=6)\n        px,py=project(data.xpos[b]); draw.ellipse((px-3,py-3,px+3,py+3),fill=color)\n    draw.text((10,10),f\'Recorded Newton trajectory | t={i*dt:.1f}s | grid 0.5 m\',fill=\'black\')\n    draw.text((10,27),\'CPU kinematic view (blue=left, green=right)\',fill=\'black\')\n    encoder.stdin.write(np.asarray(frame).tobytes())\n    if i in sample_indices:samples.append(frame.copy())\nencoder.stdin.close(); encoder.wait(timeout=30); log.close()\nassert encoder.returncode==0,video.with_suffix(\'.render.log\').read_text()\nsheet=Image.new(\'RGB\',(1280,720),\'white\')\nfor j,frame in enumerate(samples[:4]):sheet.paste(frame,((j%2)*640,(j//2)*360))\nsheet.save(path.with_suffix(\'.png\'))\nprint(\'RECORDED_TRAJECTORY_VIDEO:\',str(video),video.stat().st_size,flush=True)\n')


In [ ]:
"""Colab round runner: finite jobs, live progress, periodic verified Drive copies."""

import hashlib
import json
import os
import re
import shutil
import subprocess
import time
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo


def run_round(task, experiment, role, iterations, num_envs=256, resume_run=None,
              checkpoint=None, source_commit=None, max_seconds=2700, hydra_overrides=()):
    python = '/content/rpo-venv/bin/python'
    repo = Path('/content/LearningRL')
    lab = Path('/content/IsaacLab')
    drive = Path('/content/drive/MyDrive/LearningRL/zhaoyu-lu-HMD')
    assert Path('/content/drive/MyDrive').is_dir(), 'Mount the authorized group Drive first'
    assert Path(python).is_file(), 'Simulation environment needs recovery first'
    model = Path('/content/rpo-converted-path.txt').read_text().strip()
    assert Path(model).is_file()
    actual_commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
    if source_commit:
        assert actual_commit == source_commit, (actual_commit, source_commit)
    stamp = datetime.now(ZoneInfo('America/Chicago')).strftime('%Y-%m-%d_%H%M%S')
    name = role + '_' + stamp
    backup = drive / 'rounds' / name
    backup.mkdir(parents=True, exist_ok=False)
    local_log = Path('/content') / (name + '.log')
    env = os.environ.copy()
    env.update(OMNI_KIT_ACCEPT_EULA='YES', ISAACLAB_PATH=str(lab), MPLBACKEND='Agg',
               WANDB_MODE='disabled', PYTHONUNBUFFERED='1', LEARNINGRL_RPO_USD=model)
    args = [python, str(repo / 'LearningRLlab/scripts/train.py'), '--task', task,
            '--num_envs', str(num_envs), '--max_iterations', str(iterations),
            '--viz', 'none', '--logger', 'tensorboard', '--run_name', name,
            'physics=newton_mjwarp', 'agent.save_interval=50']
    args += list(hydra_overrides)
    if resume_run:
        assert checkpoint
        args += ['--resume', '--load_run', resume_run, '--checkpoint', checkpoint]
    metadata = dict(task=task, role=role, requested_additional_updates=iterations,
                    num_envs=num_envs, source_commit=actual_commit, resume_run=resume_run,
                    resume_checkpoint=checkpoint, hydra_overrides=list(hydra_overrides),
                    status='running', name=name)
    (backup / 'round.json').write_text(json.dumps(metadata, indent=2))
    stable = {}
    copied = {}

    def sync(run, final=False):
        if run is None:
            return
        for path in run.glob('model_*.pt'):
            signature = (path.stat().st_size, path.stat().st_mtime_ns)
            if not final and stable.get(path.name) != signature:
                stable[path.name] = signature
                continue
            if signature[0] == 0 or copied.get(path.name) == signature:
                continue
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
            temp = backup / (path.name + '.partial')
            destination = backup / path.name
            shutil.copy2(path, temp)
            if (path.stat().st_size, path.stat().st_mtime_ns) != signature:
                continue
            assert hashlib.sha256(temp.read_bytes()).hexdigest() == digest
            if destination.exists():
                assert hashlib.sha256(destination.read_bytes()).hexdigest() == digest
                temp.unlink()
            else:
                temp.replace(destination)
            copied[path.name] = signature
            print('DRIVE_CHECKPOINT_VERIFIED:', path.name, signature[0], flush=True)

    started = time.monotonic()
    run = None
    last_printed = -1
    with local_log.open('w') as log:
        process = subprocess.Popen(args, cwd=lab, env=env, stdout=log, stderr=subprocess.STDOUT)
        while process.poll() is None:
            time.sleep(10)
            candidates = list((lab / 'logs/rsl_rl' / experiment).glob('*_' + name))
            if candidates:
                run = sorted(candidates)[-1]
            sync(run)
            text = local_log.read_text(errors='replace')
            updates = re.findall(r'Learning iteration\s+(\d+)/(\d+)', text)
            count = len(updates)
            if count >= last_printed + 20:
                last_printed = count
                print(role, 'updates recorded:', count, '/', iterations, flush=True)
            if time.monotonic() - started > max_seconds:
                process.terminate()
                process.wait(timeout=20)
                metadata['status'] = 'compute_limit'
                break
    sync(run, final=True)
    text = local_log.read_text(errors='replace')
    rows = []
    for block in text.split('Learning iteration ')[1:]:
        match = re.match(r'(\d+)/(\d+)', block)
        if not match:
            continue
        row = {'iteration': int(match[1])}
        for line in block.splitlines():
            if ':' in line:
                key, value = line.rsplit(':', 1)
                try:
                    row[key.strip()] = float(value.strip())
                except ValueError:
                    pass
        rows.append(row)
    metadata.update(exit_code=process.returncode, recorded_updates=len(rows),
                    elapsed_seconds=round(time.monotonic() - started, 1),
                    checkpoints=list(copied), run_directory=str(run),
                    task_success_established=False, independent_evaluation_completed=False)
    metadata['status'] = 'completed' if process.returncode == 0 else metadata.get('status', 'failed')
    if process.returncode != 0 and metadata['status'] == 'running':
        metadata['status'] = 'failed'
    for key in ['Mean reward', 'Mean episode length', 'Metrics/base_velocity/stage',
                'Metrics/base_velocity/success', 'Metrics/base_velocity/error_vel_xy',
                'Metrics/base_velocity/error_vel_yaw']:
        values = [r[key] for r in rows if key in r]
        if values:
            metadata[key] = {'first': values[0], 'last': values[-1], 'max': max(values)}
    (backup / 'metrics.json').write_text(json.dumps(rows, indent=2))
    (backup / 'round.json').write_text(json.dumps(metadata, indent=2))
    shutil.copy2(local_log, backup / 'training.log')
    if run:
        for relative in ['params', 'git']:
            source = run / relative
            if source.is_dir():
                shutil.copytree(source, backup / relative, dirs_exist_ok=True)
    print('ROUND_RESULT:', json.dumps(metadata, indent=2), flush=True)
    if process.returncode != 0:
        print(text[-10000:], flush=True)
    assert process.returncode == 0, 'Training failed; preserved log and any checkpoints in Drive'
    assert len(rows) == iterations, 'Unexpected update count; inspect training.log'
    assert copied, 'No checkpoint was backed up'
    return metadata


In [ ]:
RUN_TRAINING = False
FORWARD_CHUNKS = 4
CONTROLLER_SOURCE = '"""Finite forward curriculum: up to four 500-update chunks with independent evaluations."""\nimport hashlib\nimport json\nimport os\nimport shutil\nimport subprocess\nfrom datetime import datetime\nfrom pathlib import Path\n\nSOURCE = \'36521e4a473595ccc08ad73fb99113f408ef4960\'\nTASK = \'LearningRL-RPO-ForwardWalk-v0\'\nEXPERIMENT = \'learningrl_rpo_forward_walk\'\nroot = Path(\'/content/drive/MyDrive/LearningRL/zhaoyu-lu-HMD\')\ncurriculum = root / \'forward_curriculum\'\ncurriculum.mkdir(parents=True, exist_ok=True)\ncheckpoint_source = root/\'rounds/walk_tracking_2026-11-16_223616/model_748.pt\'\nprevious = sorted([p for p in (root/\'rounds\').glob(\'forward_chunk*\') if list(p.glob(\'model_*.pt\'))], key=lambda p:tuple(p.name.split(\'_\')[-2:]))\nif previous:\n    checkpoint_source = max(previous[-1].glob(\'model_*.pt\'),key=lambda p:int(p.stem.split(\'_\')[-1]))\nseed_run = Path(\'/content/IsaacLab/logs/rsl_rl\')/EXPERIMENT/(\'restored_\'+checkpoint_source.parent.name)\nseed_run.mkdir(parents=True, exist_ok=True)\nshutil.copy2(checkpoint_source, seed_run/checkpoint_source.name)\nassert hashlib.sha256(checkpoint_source.read_bytes()).digest() == hashlib.sha256((seed_run/checkpoint_source.name).read_bytes()).digest()\nresume_run = seed_run.name\nresume_checkpoint = checkpoint_source.name\nresults = []\nsession_id=datetime.now().strftime(\'%Y%m%d_%H%M%S\')\nchunks=int(globals().get(\'FORWARD_CHUNKS\',4))\nweight_check = \'\'\'\nimport torch,sys,json\nm=torch.load(sys.argv[1],map_location=\'cpu\',weights_only=False)\nt=[v for v in m[\'actor_state_dict\'].values() if isinstance(v,torch.Tensor) and v.is_floating_point()]\nassert t and all(torch.isfinite(v).all() for v in t)\nprint(json.dumps({\'finite_actor\':True,\'iteration\':m.get(\'iter\')}))\n\'\'\'\nfor chunk in range(1, chunks+1):\n    run = run_round(TASK,EXPERIMENT,f\'forward_chunk{chunk}\',500,512,\n                    resume_run=resume_run,checkpoint=resume_checkpoint,\n                    source_commit=SOURCE,max_seconds=3600)\n    folder = root/\'rounds\'/run[\'name\']\n    checkpoint = max(folder.glob(\'model_*.pt\'),key=lambda p:int(p.stem.split(\'_\')[-1]))\n    check = subprocess.run([\'/content/rpo-venv/bin/python\',\'-c\',weight_check,str(checkpoint)],capture_output=True,text=True,check=True)\n    weights = json.loads(check.stdout.strip().splitlines()[-1])\n    import math\n    metrics=json.loads((folder/\'metrics.json\').read_text())\n    losses=[v for row in metrics for k,v in row.items() if \'loss\' in k.lower()]\n    assert losses and all(math.isfinite(v) for v in losses), \'Non-finite training losses\'\n    eval_env = os.environ.copy()\n    eval_env.update(OMNI_KIT_ACCEPT_EULA=\'YES\',ISAACLAB_PATH=\'/content/IsaacLab\',WANDB_MODE=\'disabled\',MPLBACKEND=\'Agg\',LEARNINGRL_RPO_USD=Path(\'/content/rpo-converted-path.txt\').read_text().strip())\n    with (folder/\'evaluation.log\').open(\'w\') as log:\n        evaluated = subprocess.run([\'/content/rpo-venv/bin/python\',\'/content/evaluate_walk.py\',\n                        \'--task\',TASK+\'-PLAY\',\'--checkpoint\',str(checkpoint),\n                        \'--output\',str(folder/\'evaluation.json\'),\'--forward-only\',\'--trace\',\n                        \'physics=newton_mjwarp\'],env=eval_env,cwd=\'/content/IsaacLab\',stdout=log,stderr=subprocess.STDOUT,timeout=600)\n    if evaluated.returncode:\n        print((folder/\'evaluation.log\').read_text()[-6000:],flush=True)\n    evaluated.check_returncode()\n    evaluation=json.loads((folder/\'evaluation.json\').read_text())\n    phase=evaluation[\'phase_mean_actual_vx_vy_wz_per_trial\'][1]\n    forward_speeds=[v[0] for v in phase]\n    passed=(evaluation[\'survived_10_seconds\']==5\n            and min(forward_speeds)>=.1\n            and max(abs(v[1]) for v in phase)<=.07\n            and max(abs(v[2]) for v in phase)<=.2\n            and max(evaluation[\'mean_xy_velocity_error_m_s\'])<=.12\n            and len(evaluation[\'net_xy_displacement_m_survivors\'])==5\n            and min(evaluation[\'net_xy_displacement_m_survivors\'])>=1.)\n    record={\'chunk\':chunk,\'additional_updates\':chunk*500,\'run\':run,\'weights\':weights,\n            \'evaluation\':evaluation,\'forward_speed_m_s\':forward_speeds,\n            \'internal_one_meter_gate_passed\':passed,\'box_transfer_success_established\':False}\n    results.append(record)\n    run.update(independent_evaluation_completed=True,internal_one_meter_gate_passed=passed)\n    (folder/\'round.json\').write_text(json.dumps(run,indent=2))\n    (folder/\'verification.json\').write_text(json.dumps(record,indent=2))\n    (curriculum/\'curriculum_results.json\').write_text(json.dumps(results,indent=2))\n    (curriculum/f\'curriculum_results_{session_id}.json\').write_text(json.dumps(results,indent=2))\n    print(\'FORWARD_CHUNK_EVALUATION:\',json.dumps({\'chunk\':chunk,\'updates\':chunk*500,\'survived\':evaluation[\'survived_10_seconds\'],\'forward_speed_m_s\':forward_speeds,\'displacement_m\':evaluation[\'net_xy_displacement_m_survivors\'],\'one_meter_gate_passed\':passed,\'drive_directory\':str(folder)},indent=2),flush=True)\n    render_env=eval_env.copy()\n    render_env[\'MUJOCO_GL\']=\'egl\'\n    rendered=subprocess.run([\'/content/rpo-venv/bin/python\',\'/content/render_trace.py\',str(folder/\'evaluation.npz\')],env=render_env,capture_output=True,text=True,timeout=180)\n    print(\'TRACE_RENDER:\',rendered.returncode,rendered.stdout[-500:],rendered.stderr[-500:],flush=True)\n    if passed:\n        print(\'FORWARD_GATE_REACHED: stop additional updates and preserve the evaluated checkpoint.\',flush=True)\n        break\n    resume_run=Path(run[\'run_directory\']).name\n    resume_checkpoint=checkpoint.name\nprint(\'FORWARD_CURRICULUM_FINISHED:\',len(results)*500,\'additional PPO updates\',flush=True)\n'
if RUN_TRAINING:
    exec(CONTROLLER_SOURCE)
else:
    print('Set RUN_TRAINING=True to continue. All checkpoints and evaluations stay in group Drive.')


## Interpretation
The first old walking policy stood still despite responding to velocity commands. Judge this curriculum by actual forward body velocity, displacement, termination, and recorded trajectories. An increase in reward alone is insufficient. The one-meter gate is an internal diagnostic, not a course grading rule or evidence of hardware readiness. Manipulation and loaded walking are not integrated.
